# ChatgaiyyaAlap MT — Day 4 (Google Colab)
Dictionary-Augmented Prompting, Both Directions

**Requires Day 1-3 done first.** This notebook pulls from your GitHub repo:
- `outputs/sampled_pairs.csv` — the same test sample used Day 2 and Day 3
- `outputs/day2_zeroshot_*.json` / `outputs/day3_fewshot_*.json` — for the running comparison

It also needs the **dictionary CSV**, which is gitignored (raw dataset, not committed),
so you'll re-provide it the same way as Day 1: upload directly, or pull from Drive.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (optional, faster).


## Step 0 — Install dependencies

In [1]:
!pip -q install transformers accelerate sacrebleu pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.0 MB/s eta 0:00:00


## Step 2 — Get the dictionary CSV

`data/` is gitignored, so the dictionary isn't in your repo. Same two options as Day 1
— run **one** of the two cells below.


In [2]:
import os

In [3]:
# --- Option A: direct upload (session-only) ---
from google.colab import files
import shutil

os.makedirs("data", exist_ok=True)
print("Select the DICTIONARY csv file:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, "data/dictionary.csv")
print("Saved to data/dictionary.csv")


Select the DICTIONARY csv file:


Saving dictionary.csv to dictionary (1).csv
Saved to data/dictionary.csv


## Step 3 — Load the test sample and the dictionary

In [4]:
import pandas as pd
from pathlib import Path

SAMPLE_PATH = Path("/content/sampled_pairs.csv")
DICTIONARY_PATH = Path("/content/data/dictionary.csv")

for p in (SAMPLE_PATH, DICTIONARY_PATH):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found — see Step 2 / earlier days.")

sample = pd.read_csv(SAMPLE_PATH)

# Same column auto-detection approach as Day 1
BANGLA_HINTS = ["bangla", "bengali", "standard"]
CHATGAIYA_HINTS = ["chittagon", "chatgaiya", "chatgaiyya", "dialect"]

def guess_columns(df, label):
    cols = list(df.columns)
    lower_cols = {c: c.lower() for c in cols}
    b_col = next((c for c, lc in lower_cols.items() if any(h in lc for h in BANGLA_HINTS)), None)
    c_col = next((c for c, lc in lower_cols.items() if any(h in lc for h in CHATGAIYA_HINTS)), None)
    if b_col is None or c_col is None:
        if len(cols) == 2:
            print(f"[WARN] Guessing columns for {label}: '{cols[0]}'=Bangla, '{cols[1]}'=Chatgaiya. VERIFY.")
            return cols[0], cols[1]
        raise ValueError(f"Could not detect columns in {label}. Columns: {cols}")
    return b_col, c_col

dict_df = pd.read_csv(DICTIONARY_PATH)
b_col, c_col = guess_columns(dict_df, "dictionary.csv")
dict_df = dict_df.rename(columns={b_col: "bangla_word", c_col: "chatgaiya_word"})
dict_df = dict_df[["bangla_word", "chatgaiya_word"]].dropna().drop_duplicates().reset_index(drop=True)

print(f"Test sample: {len(sample)} pairs")
print(f"Dictionary: {len(dict_df)} word pairs")
dict_df.head()


[WARN] Guessing columns for dictionary.csv: 'বাংলা'=Bangla, 'চট্টগ্রাম'=Chatgaiya. VERIFY.
Test sample: 60 pairs
Dictionary: 1532 word pairs


,bangla_word,chatgaiya_word
0,আমি,অ্যাঁই
1,সিন্দাবাদের,সিন্দাবাদর
2,বেধেছে,বাইদ্ধে
3,গভীর,গভীর
4,আকাশের,আকাশের


## Step 4 — Per-sentence dictionary lookup

For each test sentence, find dictionary entries whose word actually appears in that
sentence, and only inject *those* relevant mappings into the prompt — not the whole
1500-word dictionary (that would blow past context limits and dilute the signal).

This is simple substring matching. Chatgaiya has no standardized spelling, so matches
will be incomplete on that side — note that as a known limitation, not a bug.


In [5]:
def lookup_relevant_words(sentence, direction):
    """Return list of (source_word, target_word) dict entries found in `sentence`."""
    matches = []
    if direction == "b2c":
        word_col, target_col = "bangla_word", "chatgaiya_word"
    else:
        word_col, target_col = "chatgaiya_word", "bangla_word"

    for _, row in dict_df.iterrows():
        word = str(row[word_col]).strip()
        if word and word in sentence:
            matches.append((word, row[target_col]))
    return matches


# Quick sanity check on one sentence
example_sentence = sample.iloc[0]["bangla"]
example_matches = lookup_relevant_words(example_sentence, "b2c")
print(f"Sentence: {example_sentence}")
print(f"Matched {len(example_matches)} dictionary entries: {example_matches[:10]}")


Sentence: সব সময় এমন জুতা দিয়ে মাইর খায়
Matched 9 dictionary entries: [('সব', 'হক্কল'), ('সময়', 'সমত'), ('দিয়ে', 'দি\xa0\xa0'), ('মন', 'মন'), ('দিয়ে', 'দি'), ('এমন', 'এন'), ('মাইর', 'মাইর'), ('জুতা', 'জুতা'), ('খায়', 'হায়')]


## Step 5 — Build the dictionary-augmented prompt template

Same instruction framing as Day 2/3, but with a per-sentence "relevant vocabulary"
block inserted before the sentence, built dynamically from Step 4's lookup — this is
why the template is a *function*, not a static string like Day 2/3.


In [6]:
import os
os.makedirs("prompts", exist_ok=True)

def build_dict_prompt(text, direction):
    matches = lookup_relevant_words(text, direction)
    if matches:
        vocab_lines = "\n".join(f"- {src} = {tgt}" for src, tgt in matches)
        vocab_block = f"Relevant vocabulary:\n{vocab_lines}\n\n"
    else:
        vocab_block = ""   # no dictionary matches for this sentence — fall back to plain zero-shot

    if direction == "b2c":
        return (
            "Translate the following sentence from Standard Bangla into the "
            "Chittagonian (Chatgaiya) dialect spoken in southeastern Bangladesh. Use "
            "the vocabulary list below where relevant. Reply with only the translated "
            "sentence, nothing else.\n\n"
            f"{vocab_block}"
            f"Standard Bangla: {text}\n"
            "Chittagonian:"
        )
    else:
        return (
            "Translate the following sentence from the Chittagonian (Chatgaiya) "
            "dialect into Standard Bangla. Use the vocabulary list below where "
            "relevant. Reply with only the translated sentence, nothing else.\n\n"
            f"{vocab_block}"
            f"Chittagonian: {text}\n"
            "Standard Bangla:"
        )


# Save a couple of *example* rendered prompts for the repo (not a reusable static
# template, since the vocab block changes per sentence) so teammates can see the format
sample_prompt_b2c = build_dict_prompt(sample.iloc[0]["bangla"], "b2c")
sample_prompt_c2b = build_dict_prompt(sample.iloc[0]["chatgaiya"], "c2b")

with open("prompts/dictionary_b2c_example_v1.txt", "w", encoding="utf-8") as f:
    f.write(sample_prompt_b2c)
with open("prompts/dictionary_c2b_example_v1.txt", "w", encoding="utf-8") as f:
    f.write(sample_prompt_c2b)

print(sample_prompt_b2c)


Translate the following sentence from Standard Bangla into the Chittagonian (Chatgaiya) dialect spoken in southeastern Bangladesh. Use the vocabulary list below where relevant. Reply with only the translated sentence, nothing else.

Relevant vocabulary:
- সব = হক্কল
- সময় = সমত
- দিয়ে = দি  
- মন = মন
- দিয়ে = দি
- এমন = এন
- মাইর = মাইর
- জুতা = জুতা
- খায় = হায়

Standard Bangla: সব সময় এমন জুতা দিয়ে মাইর খায়
Chittagonian:


## Step 6 — Load the model

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # keep consistent with Day 1-3 unless your team agreed to change it

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print(f"Loaded on device: {device}")

GEN_CONFIG = dict(max_new_tokens=100, temperature=0.3, do_sample=True)   # same as Day 2/3
print("Generation config:", GEN_CONFIG)


Loading Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded on device: cuda
Generation config: {'max_new_tokens': 100, 'temperature': 0.3, 'do_sample': True}


## Step 7 — Run dictionary-augmented translation across the same sample, both directions

In [8]:
import time
from tqdm.auto import tqdm

def translate_with_prompt(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, **GEN_CONFIG)
    latency = time.time() - start

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text_out = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text_out, latency


def run_direction(df, direction, source_col, ref_col):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        source_text = row[source_col]
        matches = lookup_relevant_words(source_text, direction)
        prompt_text = build_dict_prompt(source_text, direction)
        prediction, latency = translate_with_prompt(prompt_text)
        rows.append({
            "source": source_text,
            "reference": row[ref_col],
            "prediction": prediction,
            "latency_sec": round(latency, 3),
            "num_dict_matches": len(matches),
            "dict_matches": matches,
        })
    return rows


print("Running dictionary-augmented: Bangla -> Chatgaiya ...")
results_b2c = run_direction(sample, "b2c", source_col="bangla", ref_col="chatgaiya")

print("Running dictionary-augmented: Chatgaiya -> Bangla ...")
results_c2b = run_direction(sample, "c2b", source_col="chatgaiya", ref_col="bangla")

avg_matches_b2c = sum(r["num_dict_matches"] for r in results_b2c) / len(results_b2c)
avg_matches_c2b = sum(r["num_dict_matches"] for r in results_c2b) / len(results_c2b)
print(f"Avg dictionary matches per sentence — B2C: {avg_matches_b2c:.1f}, C2B: {avg_matches_c2b:.1f}")
print("If these are close to 0, the dictionary coverage on this sample is thin — note that as a limitation.")


Running dictionary-augmented: Bangla -> Chatgaiya ...


  0%|          | 0/60 [00:00<?, ?it/s]

Running dictionary-augmented: Chatgaiya -> Bangla ...


  0%|          | 0/60 [00:00<?, ?it/s]

Avg dictionary matches per sentence — B2C: 7.0, C2B: 10.3
If these are close to 0, the dictionary coverage on this sample is thin — note that as a limitation.


## Step 8 — Score with BLEU and chrF

In [9]:
import sacrebleu, statistics

def score_pair(prediction, reference):
    if not prediction.strip():
        return 0.0, 0.0
    bleu = sacrebleu.sentence_bleu(prediction, [reference]).score / 100
    chrf = sacrebleu.sentence_chrf(prediction, [reference]).score / 100
    return round(bleu, 4), round(chrf, 4)


def add_scores(rows):
    for r in rows:
        bleu, chrf = score_pair(r["prediction"], r["reference"])
        r["bleu"] = bleu
        r["chrf"] = chrf
    return rows


results_b2c = add_scores(results_b2c)
results_c2b = add_scores(results_c2b)

def avg(rows, key):
    return round(statistics.mean(r[key] for r in rows), 4)

print("Bangla -> Chatgaiya  avg BLEU:", avg(results_b2c, "bleu"), " avg chrF:", avg(results_b2c, "chrf"))
print("Chatgaiya -> Bangla  avg BLEU:", avg(results_c2b, "bleu"), " avg chrF:", avg(results_c2b, "chrf"))


Bangla -> Chatgaiya  avg BLEU: 0.3781  avg chrF: 0.6096
Chatgaiya -> Bangla  avg BLEU: 0.3347  avg chrF: 0.5315


## Step 9 — Compare all three techniques so far: zero-shot vs few-shot vs dictionary


In [13]:
import json

with open("/content/day2_zeroshot_b2c.json", encoding="utf-8") as f:
    day2_b2c = json.load(f)
with open("/content/day2_zeroshot_c2b.json", encoding="utf-8") as f:
    day2_c2b = json.load(f)
with open("/content/day3_fewshot_b2c.json", encoding="utf-8") as f:
    day3_b2c = json.load(f)
with open("/content/day3_fewshot_c2b.json", encoding="utf-8") as f:
    day3_c2b = json.load(f)

rows = [
    ("B2C", "zero-shot",  day2_b2c["metrics"]["avg_bleu"], day2_b2c["metrics"]["avg_chrf"]),
    ("B2C", "few-shot",   day3_b2c["metrics"]["avg_bleu"], day3_b2c["metrics"]["avg_chrf"]),
    ("B2C", "dictionary", avg(results_b2c, "bleu"),        avg(results_b2c, "chrf")),
    ("C2B", "zero-shot",  day2_c2b["metrics"]["avg_bleu"], day2_c2b["metrics"]["avg_chrf"]),
    ("C2B", "few-shot",   day3_c2b["metrics"]["avg_bleu"], day3_c2b["metrics"]["avg_chrf"]),
    ("C2B", "dictionary", avg(results_c2b, "bleu"),        avg(results_c2b, "chrf")),
]

print(f"{'Direction':<10}{'Technique':<12}{'avg BLEU':<10}{'avg chrF':<10}")
for direction, technique, bleu, chrf in rows:
    print(f"{direction:<10}{technique:<12}{bleu:<10}{chrf:<10}")


Direction Technique   avg BLEU  avg chrF  
B2C       zero-shot   0.0415    0.1389    
B2C       few-shot    0.0646    0.2205    
B2C       dictionary  0.3781    0.6096    
C2B       zero-shot   0.0469    0.1598    
C2B       few-shot    0.097     0.2578    
C2B       dictionary  0.3347    0.5315    


## Step 10 — Spot-check and update the issue log

In [14]:
worst = sorted(results_b2c + results_c2b, key=lambda r: r["chrf"])[:5]
for r in worst:
    print(f"SRC:  {r['source']}")
    print(f"REF:  {r['reference']}")
    print(f"PRED: {r['prediction']}")
    print(f"chrF: {r['chrf']}  (dict matches used: {r['num_dict_matches']})")
    print("-" * 60)


SRC:  কোদিন হইয়্যে
REF:  কঠিন হয়েছে
PRED: বলো কৈ
chrF: 0.0244  (dict matches used: 11)
------------------------------------------------------------
SRC:  ইবে অইল অ্যাঁরার হাছে উৎসব
REF:  এটি হল আমাদের কাছে উৎসব
PRED: এটি আমরা অমনি পাই
chrF: 0.1139  (dict matches used: 12)
------------------------------------------------------------
SRC:  মুখ নিয়ে পিক পিক পাখি বলে
REF:  মুক লইয়্যে পিক পিক ফাখি কইয়্যরে
PRED: মুখ লইয়ে ফাখি ফাখি ফাখি হ
chrF: 0.1368  (dict matches used: 6)
------------------------------------------------------------
SRC:  তুঁর লগে চ্যাঁইবে আজুইন্নে
REF:  তোর সাথে দেখবে সন্ধ্যায়
PRED: তুঁর লগে চ্যাঁইবে আজুইন্নে যা
chrF: 0.1373  (dict matches used: 11)
------------------------------------------------------------
SRC:  বউরে ধরি কিলদন অইব্যু
REF:  বউকে ধরে ঘুষি দিতে হবে
PRED: বউ আসছে দিতে অই
chrF: 0.1463  (dict matches used: 11)
------------------------------------------------------------


In [15]:
issue_notes = [
    # e.g. "b2c: sentences with 0 dictionary matches scored no better than zero-shot, as expected",
    # e.g. "model sometimes inserted the dictionary target word literally even when grammatically wrong in context",
]

os.makedirs("logs", exist_ok=True)
with open("logs/issue_log.txt", "a", encoding="utf-8") as f:
    f.write(f"\n--- Day 4 dictionary-augmented ({pd.Timestamp.utcnow()}) ---\n")
    f.write(f"Avg dict matches/sentence — B2C: {avg_matches_b2c:.1f}, C2B: {avg_matches_c2b:.1f}\n")
    if issue_notes:
        for note in issue_notes:
            f.write(f"- {note}\n")
    else:
        f.write("- (fill in real observations before submitting — don't leave this empty)\n")

print("Logged to logs/issue_log.txt — go edit issue_notes above with your real findings.")


Logged to logs/issue_log.txt — go edit issue_notes above with your real findings.


## Step 11 — Save as `day4_dictionary_variant.json`

In [16]:
from datetime import datetime, timezone

output = {
    "dataset": "ChatgaiyyaAlap",
    "technique": "dictionary_augmented",
    "model": MODEL_NAME,
    "prompt_template_examples": {
        "b2c": "prompts/dictionary_b2c_example_v1.txt",
        "c2b": "prompts/dictionary_c2b_example_v1.txt",
    },
    "dictionary_size": len(dict_df),
    "generation_config": GEN_CONFIG,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "directions": {
        "bangla_to_chatgaiya": {
            "num_examples": len(results_b2c),
            "avg_dict_matches_per_sentence": round(avg_matches_b2c, 2),
            "results": results_b2c,
            "metrics": {
                "avg_bleu": avg(results_b2c, "bleu"),
                "avg_chrf": avg(results_b2c, "chrf"),
                "avg_latency_sec": round(statistics.mean(r["latency_sec"] for r in results_b2c), 3),
            },
        },
        "chatgaiya_to_bangla": {
            "num_examples": len(results_c2b),
            "avg_dict_matches_per_sentence": round(avg_matches_c2b, 2),
            "results": results_c2b,
            "metrics": {
                "avg_bleu": avg(results_c2b, "bleu"),
                "avg_chrf": avg(results_c2b, "chrf"),
                "avg_latency_sec": round(statistics.mean(r["latency_sec"] for r in results_c2b), 3),
            },
        },
    },
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/day4_dictionary_variant.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("Saved outputs/day4_dictionary_variant.json")


Saved outputs/day4_dictionary_variant.json


---
**Recap — Day 4 checklist:**
1. ✅ Loaded the dictionary and matched relevant word pairs per test sentence
2. ✅ Built a dictionary-augmented prompt variant (dynamic vocab block, both directions)
3. ✅ Ran it across the same Day 2/3 sample
4. ✅ Compared against zero-shot and few-shot, per direction
5. ✅ Saved `day4_dictionary_variant.json`

**Be ready to answer:**
- Did dictionary augmentation help most on sentences with *more* matched words, or was
  there no clear relationship? Look at that before claiming it helped or didn't.
- Any cases where the model inserted a dictionary word verbatim but grammatically wrong?
- How thin is dictionary coverage on this sample — does that limit what this technique
  can realistically show?

Next: Day 5 — full evaluation (BLEU/chrF aggregation + run-to-run consistency) across
all three techniques together.
